# Depth Pro 2: depth and normals of a picture

<a href="https://colab.research.google.com/github/Cortex78/ml-depth-pro-2/blob/claude/festive-cerf-65vitz/notebooks/depth_pro_2_picture_normals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Upload a picture and get its metric depth and its surface normals from the largest Depth Pro 2
model, **H+** (`depthpro2-h`, 872M parameters), using [Cortex78/ml-depth-pro-2](https://github.com/Cortex78/ml-depth-pro-2).

**Before you start:** choose **Runtime → Change runtime type → A100 GPU** (any NVIDIA GPU with
16 GB works for one picture), then **Runtime → Run all**. The run stops at the upload button:
choose your picture there. The first run downloads the H+ weights (3.5 GB).

What you get for each picture `NAME`:

| File | Content |
|---|---|
| `NAME-depth.png` | Depth in color: red near, blue far |
| `NAME-depth16.png` | Inverse depth as a 16-bit grayscale PNG, for compositing tools |
| `NAME-depth.npz` | `depth` in meters, `focallength_px`, `intrinsics` (3x3) and `normals` (H, W, 3) |
| `NAME-normals.png` | Normal map: red right, green up, blue toward the camera (OpenGL, Blender) |
| `NAME-points.ply` | Colored 3D point cloud with normals, in meters |

The network predicts the depth and the field of view. The normals are computed from that depth,
so they are as detailed as the depth is: fine relief that the depth does not show, as on glass or
mirrors, does not appear in the normals either.

The model weights are licensed for research only; see `LICENSE_MODEL` in the repository.

In [ ]:
# @title Settings
REPO_URL = "https://github.com/Cortex78/ml-depth-pro-2.git"  # @param {type:"string"}
BRANCH = "claude/festive-cerf-65vitz"  # @param {type:"string"}
# @markdown The largest and sharpest model. Do not change it for the best result.
MODEL = "depthpro2-h"  # @param ["depthpro2-h", "depthpro2-l", "depthpro2-b", "depthpro2-s"]
# @markdown Leave out of the point cloud what is farther than this many meters, such as sky.
MAX_DEPTH = 1000  # @param {type:"number"}
# @markdown PyTorch build: `auto` keeps PyPI's (CUDA 13) if the driver is 580 or later, else CUDA 12.6.
CUDA_BUILD = "auto"  # @param ["auto", "pypi", "cu126", "cu128", "cu130"]

## 1. GPU

In [ ]:
# @title Check the GPU
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No NVIDIA GPU in this runtime: Runtime > Change runtime type > A100 GPU.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
query = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
GPU_NAME, GPU_MEMORY, DRIVER = [
    s.strip() for s in subprocess.run(query, capture_output=True, text=True).stdout.splitlines()[0].split(",")
]
print(f"GPU: {GPU_NAME}, {GPU_MEMORY}, driver {DRIVER}")

## 2. Install

In [ ]:
# @title Install uv and the repository
import os
import shutil
from pathlib import Path

%pip install -q uv
REPO_DIR = "/content/ml-depth-pro-2"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # On a rerun, move to the branch's latest commit and undo the previous run's PyTorch edit.
    !git -C {REPO_DIR} fetch -q --depth 1 {REPO_URL} {BRANCH} && git -C {REPO_DIR} reset -q --hard FETCH_HEAD
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --format="%h %s (%cr)"

build = CUDA_BUILD
if build == "auto":
    build = "pypi" if int(DRIVER.split(".")[0]) >= 580 else "cu126"
pyproject = Path("pyproject.toml")
text = pyproject.read_text()
if build != "pypi":
    # Take torch and torchvision from the PyTorch index on Linux too, as on Windows.
    linux = "sys_platform == 'win32' or sys_platform == 'linux'"
    for old, new in [
        ("marker = \"sys_platform == 'win32'\"", f'marker = "{linux}"'),
        ("\"torchvision; sys_platform == 'win32'\"", f'"torchvision; {linux}"'),
    ]:
        if old not in text:
            raise RuntimeError(f"pyproject.toml has changed; cannot switch to {build}: {old!r} not found.")
        text = text.replace(old, new)
    text = text.replace("cu130", build)
    pyproject.write_text(text)
print(f"PyTorch build: {build} (driver {DRIVER}).")

os.environ["UV_LINK_MODE"] = "copy"  # /content and uv's cache are on different file systems.
# Colab sets MPLBACKEND to its inline backend, which only its own Python has: the commands below,
# which run in .venv, would fail to import matplotlib. The plots in this notebook use %matplotlib
# inline instead.
os.environ["MPLBACKEND"] = "Agg"
!uv sync

In [ ]:
%%writefile /content/check_env.py
import torch

from depth_pro_2.model import default_precision
from depth_pro_2.utils import default_device

print(f"torch {torch.__version__}, built for CUDA {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = default_device()
print(f"default device: {device}")
if device.type != "cuda":
    raise SystemExit("PyTorch cannot use the GPU. Set CUDA_BUILD to cu126 in Settings and rerun from the top.")
major, minor = torch.cuda.get_device_capability(device)
total = torch.cuda.get_device_properties(device).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(device)}, compute capability {major}.{minor}, {total:.0f} GiB")
print(f"bf16 supported natively: {torch.cuda.is_bf16_supported(including_emulation=False)}")
print(f"default precision: {str(default_precision(device)).removeprefix('torch.')}")

In [ ]:
# Stop here, rather than run everything on the CPU, if PyTorch cannot use the GPU.
import subprocess

check = subprocess.run(["uv", "run", "python", "/content/check_env.py"], capture_output=True, text=True)
print(check.stdout, check.stderr[-3000:] if check.returncode else "", sep="")
if check.returncode:
    raise RuntimeError("The environment check failed; see above.")

## 3. Upload a picture

Choose one or more pictures (`.jpg`, `.png`, `.heic`, `.webp`, ...). If you cancel the upload, the
repository's example photo is used instead.

In [ ]:
# @title Upload
import shlex

from google.colab import files

os.makedirs("pictures", exist_ok=True)
pictures = []
for name in files.upload():
    path = os.path.join("pictures", name)
    shutil.move(name, path)
    pictures.append(path)
if not pictures:
    pictures = ["assets/example.jpg"]
    print("No upload: using assets/example.jpg.")
print("Pictures:", pictures)

## 4. Predict depth and normals with H+

In [ ]:
# @title Predict
inputs = " ".join(f"-i {shlex.quote(p)}" for p in pictures)
!uv run depth-pro-2 predict {inputs} -o out_pictures/ -m {MODEL} --save-all --max-depth {MAX_DEPTH}

## 5. Results

For each picture: the input, the depth preview, the normal map, and the depth in meters. The last
panel stretches the colors over the picture's own depth range (2nd to 98th percentile), which shows
relief that the preview's fixed scale can hide.

In [ ]:
# @title Show
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps

for picture in pictures:
    stem = Path(picture).stem
    data = np.load(f"out_pictures/{stem}-depth.npz")
    depth, focal, normals = data["depth"], float(data["focallength_px"]), data["normals"]
    finite = depth[np.isfinite(depth)]
    print(f"{picture}: {depth.shape[1]}x{depth.shape[0]}, focal length {focal:.1f} px")
    print(f"  depth {finite.min():.2f} m to {finite.max():.2f} m, median {np.median(finite):.2f} m; "
          f"{100 * (1 - finite.size / depth.size):.1f}% infinitely far; "
          f"{100 * (~normals.any(-1)).mean():.1f}% without a normal")

    low, high = np.percentile(finite, [2, 98])
    panels = [
        (ImageOps.exif_transpose(Image.open(picture)).convert("RGB"), "input", None),
        (Image.open(f"out_pictures/{stem}-depth.png"), "depth preview (red near, blue far)", None),
        (Image.open(f"out_pictures/{stem}-normals.png"),
         "normals (red right, green up, blue toward the camera)", None),
        (np.clip(depth, low, high), f"depth in meters, {low:.2f} to {high:.2f} m", "turbo_r"),
    ]
    wide = depth.shape[1] >= 1.6 * depth.shape[0]
    rows, cols = (4, 1) if wide else (2, 2)
    size = (16, 16 * rows * depth.shape[0] / depth.shape[1] + 2) if wide else (18, 13)
    fig, axes = plt.subplots(rows, cols, figsize=size)
    for ax, (image, title, cmap) in zip(np.ravel(axes), panels):
        shown = ax.imshow(image, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
        if cmap:
            fig.colorbar(shown, ax=ax, label="m", fraction=0.025)
    plt.tight_layout()
    plt.show()
!ls -lh out_pictures/

## 6. Download

In [ ]:
# @title Zip and download every output
from google.colab import files

!rm -f /content/depth_pro_2_pictures.zip
!zip -qr /content/depth_pro_2_pictures.zip out_pictures
!ls -lh /content/depth_pro_2_pictures.zip
files.download("/content/depth_pro_2_pictures.zip")